In [19]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact, interaction, HBox, VBox
%matplotlib inline

In [20]:
# !pip install ipywidgets

In [21]:
df = pd.read_csv('bungalow_los_raw_data.csv')
bung_2025_2026 = pd.read_csv('bungalow_los_raw_data_2025_2026.csv')
# df.shape

In [22]:
# subset of los data without duplicates
# remove prior stat and previous kennel to elimiates duplicates
df2 = df.drop(columns=['prior_kennel', 'prior_stamp_1', 'time_in_previous_kennel']).copy()
df2 = df2.drop_duplicates()

In [23]:
#create a flag for rows which are adoptiosn return true or false
df2['is_adoption'] = (df2['outcome_type'].astype(str).str.upper() == 'ADOPTION')
#create a flag for rows which are euth return true or false
df2['is_euth'] =  (df2['outcome_type'].astype(str).str.upper() =='EUTH')
# create a flag for reows which are transfer, return true or false
df2['is_transfer'] = (df2['outcome_type'].astype(str).str.upper() == 'TRANSFER')
# create a flag for rows which are fta, return true or false
df2['is_fta'] = (df2['outcome_type'].astype(str).str.upper() == 'FOSTER')

In [24]:
# group all 4 tiers and calculate both the average and los and the adoption rate
grouped = df2.groupby(['outcome_year', 'outcome_type', 'outcome_subtype', 'kennel_no'])
metrics = grouped.agg(
    mean_los=('los', 'mean'),
    is_adopt=('is_adoption', 'sum'),
    is_euth=('is_euth', 'sum'),
    is_transfer=('is_transfer', 'sum'),
    is_fta=('is_fta', 'sum'),
    total_animals=('is_adoption', 'count')
).reset_index()


In [25]:
# build multi tier dictionary
kennel_year_dic = {}
for _, row in metrics.iterrows():
    year_str = str(row['outcome_year']).strip().replace('.0', '')
    outcome_str = str(row['outcome_type']).strip().lower()
    o_sub_str =str(row['outcome_subtype']).strip().lower()
    kennel_str = str(row['kennel_no']).strip().lower()
    # initialize year if it doesn't exists    
    if year_str not in kennel_year_dic:
        kennel_year_dic[year_str] = {}
    # check if outcojme type exists wihting the specfic year dictionary
    if outcome_str not in kennel_year_dic[year_str]:
        kennel_year_dic[year_str][outcome_str] = {}
    # check if subtype exists within the specidic outcome
    if o_sub_str not in kennel_year_dic[year_str][outcome_str]:
        kennel_year_dic[year_str][outcome_str][o_sub_str] = {}
    # assign the value
    kennel_year_dic[year_str][outcome_str][o_sub_str][kennel_str] = {
        'los': float(row['mean_los']),
        'adopt_count' : int(row['is_adopt']),
        'euth_count' : int(row['is_euth']),
        'transfer_count' : int(row['is_transfer']),
        'fta_count' : int(row['is_fta']),
        'total_count' : int(row['total_animals'])
    }
ds01 =  kennel_year_dic['2025']['adoption']['new'].get('ds03')


In [26]:
# Extract an ordered list of available year for the dropdown menu
available_years = sorted(list(kennel_year_dic.keys()), reverse=True)
available_outcome = sorted(df2['outcome_type'].dropna().astype(str).unique().tolist())
available_subtypes = sorted(df2['outcome_subtype'].dropna().astype(str).unique().tolist())

In [30]:
# define function tha draws the entire graph dynamically based on interactive inputs








def update_kennel_map(target_year=available_years, target_outcome=available_outcome, target_subtype=available_subtypes, display_metric='Length of Stay', color_theme='YlOrRd', text_size=7, show_grid_lines=True, max_stay_filter=60.0, search_kennel=''):
    global active_year_dic
    combined_kennel_data = {}
    
    # 1. Standardize widget selections to lowercase and remove spaces
    selected_years = [str(y).strip().replace('.0', '') for y in (target_year if not isinstance(target_year, (str, int)) else [target_year])]
    selected_outcomes = [str(o).strip().lower() for o in (target_outcome if not isinstance(target_outcome, (str, int)) else [target_outcome])]
    selected_subtypes = [str(s).strip().lower() for s in (target_subtype if not isinstance(target_subtype, (str, int)) else [target_subtype])]
    
    # 2. Case-insensitive dictionary matching loop
    for yr_key, year_dic in kennel_year_dic.items():
        if str(yr_key).strip().replace('.0', '') in selected_years:
            for outc_key, outcome_dic in year_dic.items():
                if str(outc_key).strip().lower() in selected_outcomes:
                    for subc_key, subtype_dic in outcome_dic.items():
                        if str(subc_key).strip().lower() in selected_subtypes:
                            for kennel, val_dict in subtype_dic.items():
                                # strict type enforcment: ensure its a vadid metirc dictionary
                                if not isinstance(val_dict, dict):
                                    continue
                                if kennel not in combined_kennel_data:
                                    combined_kennel_data[kennel] = {'los_list':[], 'adopt_sum': 0 , 'euth_sum':0, 'transfer_sum':0, 'fta_sum':0, 'total_sum':0}
                                combined_kennel_data[kennel]['los_list'].append(val_dict['los'])
                                combined_kennel_data[kennel]['adopt_sum'] += val_dict['adopt_count']
                                combined_kennel_data[kennel]['euth_sum'] += val_dict['euth_count']
                                combined_kennel_data[kennel]['transfer_sum'] += val_dict['transfer_count']
                                combined_kennel_data[kennel]['fta_sum'] += val_dict['fta_count']
                                combined_kennel_data[kennel]['total_sum'] += val_dict['total_count']
    
    # 3. Average the collected values for each kennel
    active_year_dic = {}
    for kennel, data in combined_kennel_data.items():
        if display_metric == 'Length of Stay':
            if data['los_list']:
                active_year_dic[kennel] = np.mean(data['los_list'])
        elif display_metric == 'Adoption Rate': # adopt rate path
            if data['total_sum'] > 0:
                # calculate percentage 
                active_year_dic[kennel] = (data['adopt_sum'] / data['total_sum']) * 100.0
        else:
            if data['euth_sum']:
                active_year_dic[kennel] = data['euth_sum']
            if data['transfer_sum']:
                active_year_dic[kennel] = data['transfer_sum']
            if data['adopt_sum']:
                active_year_dic[kennel] = data['adopt_sum']
            if data['fta_sum']:
                active_year_dic[kennel] = data['fta_sum']


    # get_value and label innner functions
    def get_value_and_label(kennel):
        if kennel == '' or pd.isna(kennel): 
            return np.nan, ''
        kennel_key = str(kennel).strip()
        v = active_year_dic.get(kennel, np.nan)
        
        # max stay slider should only filter when looking at Length of Stay
        if display_metric == 'Length of Stay' and not np.isnan(v) and v > max_stay_filter:
            return np.nan, ''
        if np.isnan(v):
            lbl = f"{kennel_key} \n--"
        elif display_metric == 'Length of Stay':
            lbl = f"{kennel_key} \n{v:.1f}" 
        elif display_metric == 'Adoption Rate':
            lbl = f"{kennel_key} \n {v:.1f}%"
        else:
            lbl = f"{kennel_key}\n{int(v) if isinstance(v, (int, float)) and not np.isnan(v) else v}"
        
        if search_kennel and str(search_kennel) == kennel:
            lbl = f"⭐\n{kennel}\n{v:.1f}"
        return v, lbl
    # Define physical layout dimentions for frist section
    
    rows_a = ['Row 1', 'Row 2', 'Row 3', 'Row 4', 'Row 5', 'Row 6']
    cols_a = ['Suites','','4B','6B','','8B','10B','','12B','14B','','16B','18B','','20B','22B']
    
    # Map sequential kennel numbers directly to coordinate positions for first section
    # Maps kennels 5001 - to col 1, etc
    grid_kennels_a = [
        ['ds01','','5001','5024','','5025','5048','','5049','5072','','5073','5096','','5097','5120'],
        ['ds02','','5002','5023','','5026','5047','','5050','5071','','5074','5095','','5098','5119'],
        ['ds03','','5003','5022','','5027','5046','','5051','5070','','5075','5094','','5099','5118'],
        # ['','','', '','','','','','','','','','',''],
        ['','','5004','5021','','5028','5045','','5052','5069','','5076','5093','','5100','5117'],
        ['','','5005','5020','','5029','5044','','5053','5068','','5077','5092','','5101','5116'],
        ['','','5006','5019','','5030','5043','','5054','5067','','5078','5091','','5102','5115']
    ]
    
    # Fill the grid with the dictionary values (defualt to 0 if a kennel number is empty
    grid_values_a = [[get_value_and_label(k)[0] for k in row] for row in grid_kennels_a]
    # grid_values_a = [[kennel_dic.get(kennel,np.nan) if kennel != ''  else np.nan for kennel in row] for row in grid_kennels_a]
    df_a = pd.DataFrame(grid_values_a, index=rows_a, columns= cols_a)
    mask_a = pd.DataFrame(grid_kennels_a, index=rows_a, columns=cols_a) == ''
    # annotate each cell with both the kennel number and its average los
    labels_a = np.array([[get_value_and_label(k)[1] for k in row] for row in grid_kennels_a])
    # labels_a = np.array([[f"{k}\n{v:.1f}" for k, v in zip(r_k, r_v)]
    #                    for r_k, r_v in zip(grid_kennels_a, grid_values_a)])
    
    
    # Define physical layout dimentions for second section
    
    rows_b = ['Row 7', 'Row 8', 'Row 9', 'Row 10', 'Row 11', 'Row 12']
    cols_b = ['      ','','4A','6A','','8A','10A','','12A','14A','','16A','18A','','20A','22A']
    
    # Map sequential kennel numbers directly to coordinate positions for first section
    # Maps kennels 5001 - to col 1, etc
    grid_kennels_b = [
        ['','','5007','5018','','5031','5042','','5055','5066','','5079','5090','','5103','5114'],
        ['','','5008','5017','','5032','5041','','5056','5065','','5080','5089','','5104','5113'],
        ['','','5009','5016','','5033','5040','','5057','5064','','5081','5088','','5105','5112'],
        # ['','','', '','','','','','','','','','',''],
        ['','','5010','5015','','5034','5039','','5058','5063','','5082','5087','','5106','5111'],
        ['','','5011','5014','','5035','5038','','5059','5062','','5083','5086','','5107','5110'],
        ['','','5012','5013','','5036','5037','','5060','5061','','5084','5085','','5108','5109']
    ]
    
    # Fill the grid with the dictionary values (defualt to 0 if a kennel number is empty
    grid_values_b = [[get_value_and_label(k)[0] for k in row] for row in grid_kennels_b]
    df_b = pd.DataFrame(grid_values_b, index=rows_b, columns= cols_b)
    mask_b = df_b.isna()
    # annotate each cell with both the kennel number and its average los
    labels_b = np.array([[get_value_and_label(k)[1] for k in row] for row in grid_kennels_b])
    
    # Define physical layout dimentions for third section
    
    rows_c = ['Row 13', 'Row 14', 'Row 15', 'Row 16', 'Row 17', 'Row 18']
    cols_c = ['1A','2A','','3A','5A','','7A','9A','','11A','13A','','15A','17A','','19A','21A']

    # Map sequential kennel numbers directly to coordinate positions for first section
    # Maps kennels 5001 - to col 1, etc
    grid_kennels_c = [
        ['5121','5144','','5145','5168','','5169','5192','','5193','5216','','5217','5240','','5241','5264'],
        ['5122','5143','','5146','5167','','5170','5191','','5194','5215','','5218','5239','','5242','5263'],
        ['5123','5142','','5147','5166','','5171','5190','','5195','5214','','5219','5238','','5243','5262'],
        # ['','','', '','','','','','','','','','',''],
        ['5124','5141','','5148','5165','','5172','5189','','5196','5213','','5220','5237','','5244','5261'],
        ['5125','5140','','5149','5164','','5173','5188','','5197','5212','','5221','5236','','5245','5260'],
        ['5126','5139','','5150','5163','','5174','5187','','5198','5211','','5222','5235','','5246','5259']
    ]
    
    # Fill the grid with the dictionary values (defualt to 0 if a kennel number is empty
    grid_values_c = [[get_value_and_label(k)[0] for k in row] for row in grid_kennels_c]
    df_c = pd.DataFrame(grid_values_c, index=rows_c, columns= cols_c)
    mask_c = df_c.isna()
    # annotate each cell with both the kennel number and its average los
    labels_c = np.array([[get_value_and_label(k)[1] for k in row] for row in grid_kennels_c])
    
    # Define physical layout dimentions for fourth section
    
    rows_d = ['Row 19', 'Row 20', 'Row 21', 'Row 22', 'Row 23', 'Row 24']
    cols_d = ['1B','2B','','3B','5B','','7B','9B','','11B','13B','','15B','17B','','19B','21B']
    
    # Map sequential kennel numbers directly to coordinate positions for first section
    # Maps kennels 5001 - to col 1, etc
    grid_kennels_d = [
        ['5127','5138','','5151','5162','','5175','5186','','5199','5210','','5223','5234','','5247','5258'],
        ['5128','5137','','5152','5161','','5176','5185','','5200','5209','','5224','5233','','5248','5257'],
        ['5129','5136','','5153','5160','','5177','5184','','5201','5208','','5225','5232','','5249','5256'],
        # ['','','', '','','','','','','','','','','']
        ['5130','5135','','5154','5159','','5178','5183','','5202','5207','','5226','5231','','5250','5255'],
        ['5131','5134','','5155','5158','','5179','5182','','5203','5206','','5227','5230','','5251','5254'],
        ['5132','5133','','5156','5157','','5180','5181','','5204','5205','','5228','5229','','5252','5253']
    ]
    
    # Fill the grid with the dictionary values (defualt to 0 if a kennel number is empty
    grid_values_d = [[get_value_and_label(k)[0] for k in row] for row in grid_kennels_d]
    df_d = pd.DataFrame(grid_values_d, index=rows_d, columns= cols_d)
    mask_d = df_d.isna()
    # annotate each cell with both the kennel number and its average los
    labels_d = np.array([[get_value_and_label(k)[1] for k in row] for row in grid_kennels_d])
    
    
    # Pltting the combibed stacked graph
    
    # create subpots stacked vertically 
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(nrows=4, ncols=1, figsize=(12,15))
    linewidth = 0.5 if show_grid_lines else 0
    
    # find global limits so both tiers share an identical color grading scale
    all_flat_values = [v for r in grid_values_b + grid_values_a + grid_values_c + grid_values_d for v in r if not np.isnan(v)]
    if display_metric == 'Adoption Rate':
        # hardcodee boundaratires 
        vmin = 0.0
        vmax = 100.0
    else:
        vmin = min(all_flat_values) if all_flat_values else 0
        vmax = max(all_flat_values) if all_flat_values else 10
    
    # plot top heatmap
    sns.heatmap(
        df_a,
        annot=labels_a,
        fmt="",
        cmap= color_theme, # Yellow to orange to read gradient
        linewidths=linewidth,
        mask=mask_a,
        square=True, # forces cell boxes to be smaller
        vmin=vmin,
        vmax=vmax,
        ax=ax1,
        cbar=False,
        annot_kws={'size': text_size},  # shringks text size inside boxes
        cbar_kws={'label': "Average Length of Stay (Days)", 'shrink': 0.8} # shrink color bar to match
    )
    
    # move x axis tick label to top
    ax1.xaxis.tick_top()
    # move the x axis title to the top
    ax1.xaxis.set_label_position('top')
    # remove row numbers
    ax1.set_yticklabels([])
    # remove row labels
    ax1.set_ylabel("")
    
    
    # plot second heatmap
    sns.heatmap(
        df_b,
        annot=labels_b,
        fmt="",
        cmap= color_theme, # Yellow to orange to read gradient
        linewidths=linewidth,
        mask=mask_b,
        square=True, # forces cell boxes to be smaller
        vmin=vmin,
        vmax=vmax,
        ax=ax2,
        cbar=False,
        annot_kws={'size': text_size},  # shringks text size inside boxes
        cbar_kws={'label': "Average Length of Stay (Days)", 'shrink': 0.8} # shrink color bar to match
    )
    
    # move x axis tick label to top
    ax2.xaxis.tick_bottom()
    # move the x axis title to the top
    ax2.xaxis.set_label_position('bottom')
    # remove row numbers
    ax2.set_yticklabels([])
    # remove row labels
    ax2.set_ylabel("")
    
    
    # plot third heatmap
    sns.heatmap(
        df_c,
        annot=labels_c,
        fmt="",
        cmap= color_theme, # Yellow to orange to read gradient
        linewidths=linewidth,
        mask=mask_c,
        square=True, # forces cell boxes to be smaller
        vmin=vmin,
        vmax=vmax,
        ax=ax3,
        cbar=False,
        annot_kws={'size': text_size},  # shringks text size inside boxes
        cbar_kws={'label': "Average Length of Stay (Days)", 'shrink': 0.8} # shrink color bar to match
    )
    
    # move x axis tick label to top
    ax3.xaxis.tick_top()
    # move the x axis title to the top
    ax3.xaxis.set_label_position('top')
    # remove row numbers
    ax3.set_yticklabels([])
    # remove row labels
    ax3.set_ylabel("")


    # plot last heatmap
    im = sns.heatmap(
        df_d,
        annot=labels_d,
        fmt="",
        cmap= color_theme, # Yellow to orange to read gradient
        linewidths=linewidth,
        mask=mask_d,
        square=True, # forces cell boxes to be smaller
        vmin=vmin,
        vmax=vmax,
        ax=ax4,
        cbar=False,
        annot_kws={'size': text_size},  # shringks text size inside boxes
        cbar_kws={'label': "Average Length of Stay (Days)", 'shrink': 0.8} # shrink color bar to match
    )
    
    # move x axis tick label to bottom
    ax4.xaxis.tick_bottom()
    # move the x axis title to the top
    ax4.xaxis.set_label_position('bottom')
    # remove row numbers
    ax4.set_yticklabels([])
    # remove row labels
    ax4.set_ylabel("")
    
    # Shared colorbar and title adjustments
    # define titles for each possible heatmap
    main_titles = {
        'Length of Stay' : 'Average Length of Stay by Kennel',
        'Adoption Rate' : 'Adoption Rate by Kennel',
        'Outcome Count' : 'Total Count by Kennel'
    }
    
    # define titles for each possible metric
    metric_labels = {
        'Length of Stay' : 'Average Length of Stay (Days)',
        'Adoption Rate' : 'Adoption Rate (%)',
        'Outcome Count' : 'Total Outcome Count'
    }

    # get correct title dynamically 
    current_title = main_titles.get(display_metric, 'Kennel Heatmap Dashboard')
    # get the correct title dynamically (defaul tot value if missing)
    colorbar_label = metric_labels.get(display_metric, 'Value')

    # create space on the right side to render a unified scale bar
    fig.subplots_adjust(right=0.85, hspace=0.3)
    cbar_ax = fig.add_axes([0.88, 0.2, 0.02, 0.6])
    fig.colorbar(im.collections[0], cax=cbar_ax, label=colorbar_label)
    fig.suptitle(current_title, fontsize=14, y=0.96, fontweight='bold')            
    # plt.title("Kennel Occupancy and Average Length of Stay Heatmap", fontsize=14, pad=30)
    # plt.ylabel("Vertical Row Order")
    # plt.xlabel("Facility Bank Sections", labelpad=15)
    # plt.tight_layout()
    plt.show()

# define the individual widgets
year_widget=widgets.SelectMultiple(
        options=available_years,
        value=tuple(available_years), 
        description='Select Years',
        disabled=False
)

outcome_widget=widgets.SelectMultiple(
    options=available_outcome,
    values=tuple(available_outcome),
    description='Outcome Type'
)
subtype_widget=widgets.SelectMultiple(
    options=available_subtypes,
    values=tuple(available_subtypes),
    description='Outcome Subtype'
)

theme_widget=widgets.Dropdown(
    options=['YlOrRd', 'viridis', 'plasma', 'coolwarm', 'Blues'],
    value='YlOrRd',
    description='Color Theme',
    disabled=False
)

text_widget=widgets.FloatSlider(
    options=(5,12,1),
    value=7,
    description='Text Size',
    disabled=False
)

grid_widget=widgets.Checkbox(
    value=True,
    description='Show Grid Lines'
)

stay_widget=widgets.FloatSlider(
    options=(1.0, 60.0, 0.5),
    value=60.0,
    description='May Days',
    disabled=False
)
search_widget=widgets.Text(
    value='',
    placeholder='Type Kennel Number',
    description='Search Kennel'
)
metric_widget=widgets.Dropdown(
    options=['Length of Stay', 'Adoption Rate', 'Outcome Count'],
    value='Length of Stay',
    description='Map Metric'
)
# bind the wiedges to the map function using interactive
ui_controls = widgets.interactive(
    update_kennel_map,
    target_year=year_widget,
    target_outcome=outcome_widget,
    target_subtype=subtype_widget,
    display_metric=metric_widget,
    color_theme=theme_widget,
    text_size=text_widget,
    show_grid_lines=grid_widget,
    max_stay_filter=stay_widget,
    search_kennel=search_widget
)

# create clean row for selection list
selection_row = widgets.HBox([year_widget, outcome_widget, subtype_widget])

# group the remaining widgets
settings_grid = widgets.VBox([
    widgets.HBox([metric_widget, theme_widget, text_widget]),
    widgets.HBox([grid_widget, stay_widget, search_widget])
])

# Arrange everything vertically and display the interface alongside the map output
final_dashboard = widgets.VBox([selection_row, settings_grid, ui_controls.children[-1]])
display(final_dashboard)